### Purpose

안전한 추가 API (docs/GLOBALDB_SAFE_CREATE_API.md) 배포.
Lambda `dsglobaldbCreate` 를 만들고 API Gateway `jyipsj28s9` 에 연결한다.

| 경로 | 동작 |
|---|---|
| `POST /dschemical/create` | 약품 새로 추가 (코드는 서버가 정함, 덮어쓰기 없음) |
| `GET /dschemical/next-code?infoL1=&name=` | 다음 약품 코드 미리보기 |
| `POST /dscustomer/create` | 거래처 새로 추가 (법인 중복 409, 개인 P##### 순번) |

- 기존 `/dschemical`, `/dscustomer` 의 GET/POST/PUT/DELETE 는 건드리지 않는다
- 인증 없음 (기존 API 와 같음), 패스스루 매핑 템플릿, CORS `*`, 역할 `DSmapLambda`
- 배포 전에 테스트: `pytest AWS/Lambda/dsglobaldbCreate` (moto, 실제 AWS 안 씀)
- **마지막 셀(`dev` 스테이지 배포)은 API 전체를 다시 배포한다** — 콘솔에서 배포 안 한 변경이 있으면 같이 반영됨

In [ ]:
# Norton 등이 HTTPS를 가로채는 PC에서 SSL 오류 방지 - boto3 보다 먼저 적용해야 함
import truststore
truststore.inject_into_ssl()

import json
import boto3

boto3.setup_default_session(profile_name='default', region_name='us-east-1')
from deploy_helpers import (deploy_lambda, ensure_resource_path, get_request_template, setup_lambda_method,
                            setup_options, grant_invoke, deploy_stage_and_check)

Settings

In [ ]:
FUNCTION_NAME = 'dsglobaldbCreate'
SOURCE_FILE = './dsglobaldbCreate/lambda_function.py'
ROLE_ARN = 'arn:aws:iam::199525800117:role/DSmapLambda'   # dynamodb:* on table/*, dsbaseinfo 읽기 권한 있음

API_ID = 'jyipsj28s9'
STAGE = 'dev'
ROUTES = {                      # 경로 → 메서드
    'dschemical/create': ['POST'],
    'dschemical/next-code': ['GET'],
    'dscustomer/create': ['POST'],
}
TEMPLATE_FROM = '/dschemical'

ACCOUNT_ID = ROLE_ARN.split(':')[4]

1. Lambda

In [ ]:
function_arn = deploy_lambda(FUNCTION_NAME, SOURCE_FILE, ROLE_ARN,
                             description='글로벌 DB 안전한 추가 (약품·거래처 create)', timeout=29, memory=256)
function_arn

2. API Gateway 경로 + 권한

In [ ]:
request_template = get_request_template(API_ID, TEMPLATE_FROM)

for path, methods in ROUTES.items():
    resource_id = ensure_resource_path(API_ID, path)
    for m in methods:
        setup_lambda_method(API_ID, resource_id, m, function_arn, request_template)
    setup_options(API_ID, resource_id, methods)
    grant_invoke(FUNCTION_NAME, API_ID, ACCOUNT_ID, path)

3. `dev` 스테이지 배포 + 확인 (next-code 미리보기, 쓰기 없음)

In [ ]:
res_ = deploy_stage_and_check(API_ID, STAGE, 'dschemical/next-code?infoL1=%EC%82%B4%EA%B7%A0%EC%A0%9C&name=%EB%AA%AC%EC%B9%B4%ED%8A%B8',
                              'add /dschemical/create, /dschemical/next-code, /dscustomer/create')
print('statusCode', res_['statusCode'], json.loads(res_['body']))   # 살균제·몬카트 → A11046